In [ ]:
from dataclasses import dataclass, field
import torch

@dataclass
class Config:
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    latent_dim: int = 512
    image_size: int = 256
    batch_size: int = 4
    lr: float = 0.0002
    num_epochs: int = 10
    clip_model: str = "ViT-B/32"

    # какие блоки StyleGAN размораживать (по именам слоёв)
    # пример: можно размораживать только верхние (toRGB) или только промежуточные
    trainable_blocks: list = field(default_factory=lambda: [
        "conv1", "conv2", "to_rgb"  # пример; позже ты будешь менять этот список
    ])

    source_prompt: str = "photo of a face"
    target_prompt: str = "anime style face"

    log_interval: int = 50
    save_interval: int = 500

### Install StyleGAN2-ADA-PyTorch
First, we need to install the `stylegan2_ada_pytorch` library, which contains the necessary components for loading and working with StyleGAN models.

In [ ]:
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} installed successfully.")

install_package("torch_utils")
install_package("click")
install_package("ninja")

# Clone the repository if it doesn't exist
import os
if not os.path.exists('stylegan2-ada-pytorch'):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

### Initialize StyleGAN Model
Now, let's define a function to initialize the StyleGAN generator model based on the `Config` dataclass. This function will load a pre-trained StyleGAN model (e.g., FFHQ) and move it to the specified device.

In [ ]:
import torch
import sys
import os

sys.path.insert(0, './stylegan2-ada-pytorch')

# Corrected imports for stylegan2-ada-pytorch
import dnnlib
import legacy

def load_stylegan_model(config: Config):
    print(f"Loading StyleGAN2-ADA model on device: {config.device}")

    # Example: Use a pre-trained FFHQ model
    # You can change this path to your desired pre-trained model (.pkl file)
    # If you don't have a local path, you can use a URL which 'legacy.load_network_pkl' can handle.
    # For demonstration, we'll assume a common pre-trained model URL.
    network_pkl_url = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"

    print(f"Downloading/Loading network from: {network_pkl_url}")
    with dnnlib.util.open_url(network_pkl_url) as f:
        G = legacy.load_network_pkl(f)['G_ema'] # Typically, 'G_ema' is the trained generator
    print("StyleGAN model loaded successfully.")

    G.eval().to(config.device)

    # Freeze layers not specified in trainable_blocks
    for name, param in G.named_parameters():
        param.requires_grad = False
        for block_name in config.trainable_blocks:
            if block_name in name:
                param.requires_grad = True
                print(f"Unfreezing parameter: {name}")
                break

    # Set device for the model
    G.to(config.device)

    return G

### Demonstrate StyleGAN Model Initialization
Let's create an instance of our `Config` and use the `load_stylegan_model` function to initialize the generator.

In [ ]:
config = Config()
stylegan_generator = load_stylegan_model(config)
print(f"Generator device: {next(stylegan_generator.parameters()).device}")
print(f"Number of trainable parameters in generator: {sum(p.numel() for p in stylegan_generator.parameters() if p.requires_grad)}")

### Setting up Project Structure and Training Script

To integrate the provided training loop, we'll organize the code into a `src` directory, containing `config.py` (for the `Config` dataclass) and `model_setup.py` (for the model loading and setup functions). Then, the main training script will be added.

In [ ]:
# Create the 'src' directory and install the 'clip' library and stylegan2-pytorch
!mkdir -p src
!pip install clip
!pip install stylegan2-pytorch
!pip install gdown

In [ ]:
%%writefile src/config.py
from dataclasses import dataclass, field
import torch

@dataclass
class Config:
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    latent_dim: int = 512
    image_size: int = 256
    batch_size: int = 4
    lr: float = 0.0002
    num_epochs: int = 10
    clip_model: str = "ViT-B/32"

    # какие блоки StyleGAN размораживать (по именам слоёв)
    # пример: можно размораживать только верхние (toRGB) или только промежуточные
    trainable_blocks: list = field(default_factory=lambda: [
        "conv1", "conv2", "to_rgb"  # пример; позже ты будешь менять этот список
    ])

    source_prompt: str = "photo of a face"
    target_prompt: str = "anime style face"

    log_interval: int = 50
    save_interval: int = 500

In [ ]:
%%writefile src/model_setup.py
import torch
import clip
from torchvision import transforms
from src.config import Config # Import Config from the new src/config.py
import os

def load_clip(cfg: Config):
    model, preprocess = clip.load(cfg.clip_model, device=cfg.device, jit=False)
    model.eval()
    return model, preprocess

def get_clip_embeddings(model, preprocess, texts=None, images=None):
    if texts is not None:
        tokens = clip.tokenize(texts).to(model.device)
        with torch.no_grad():
            text_features = model.encode_text(tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)
        return text_features
    if images is not None:
        # images: B x C x H x W, значения [0, 1]
        norm_images = preprocess(images).to(model.device)  # тут может потребоваться цикл
        with torch.no_grad():
            image_features = model.encode_image(norm_images)
            image_features /= image_features.norm(dim=-1, keepdim=True)
        return image_features

# Здесь ты подгружаешь StyleGAN-2.
# Для простоты возьмём готовый репозиторий stylegan2-pytorch (github.com/rosinality/stylegan2-pytorch)
# и импортируем Generator.
def load_stylegan(cfg: Config, weights_path: str):
    # stylegan2-pytorch is now installed in a separate cell
    # Ensure the correct import path for Generator
    from stylegan2_pytorch.stylegan2_pytorch import Generator

    # Handle placeholder dummy weights by downloading a real checkpoint
    if weights_path == "./checkpoints/dummy_weights.pth":
        print("Placeholder weights detected. Attempting to download a pre-trained StyleGAN2-PyTorch FFHQ checkpoint.")
        checkpoint_dir = os.path.dirname(weights_path)
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Google Drive ID for rosinality/stylegan2-pytorch FFHQ checkpoint (from their README)
        gdrive_id = "1_0wQf2hR44h003bYl4_gK9C1pL5P0nS8" # ffhq.pt
        actual_weights_filename = "ffhq_stylegan2_pytorch.pt"
        actual_weights_path = os.path.join(checkpoint_dir, actual_weights_filename)

        if not os.path.exists(actual_weights_path):
            print(f"Downloading {actual_weights_filename} from Google Drive...")
            try:
                import gdown
                gdown.download(id=gdrive_id, output=actual_weights_path, quiet=False)
                print(f"Downloaded {actual_weights_filename} successfully.")
            except ImportError:
                print("gdown not installed. Please install it with 'pip install gdown' or manually download the checkpoint.")
                raise
            except Exception as e:
                print(f"Error downloading checkpoint: {e}. Please ensure gdown is installed and the ID is correct.")
                raise
        else:
            print(f"Pre-trained checkpoint {actual_weights_filename} already exists. Using it.")

        weights_path = actual_weights_path # Update the weights_path to the downloaded file

    G = Generator(cfg.image_size, cfg.latent_dim, 8).to(cfg.device)
    ckpt = torch.load(weights_path, map_location=cfg.device)
    G.load_state_dict(ckpt["g_ema"])
    G.eval()
    return G

def setup_models(cfg: Config, sg_weights: str):
    device = cfg.device
    G_frozen = load_stylegan(cfg, sg_weights)
    G_train = load_stylegan(cfg, sg_weights)

    # Заморозим все параметры G_frozen
    for p in G_frozen.parameters():
        p.requires_grad = False

    # Разморозим только нужные блоки в G_train
    def freeze_all(m):
        for p in m.parameters():
            p.requires_grad = False
    freeze_all(G_train)

    # Тут нужно пройтись по модулям и включить grad для нужных имён.
    # Это упрощённый пример; в реальности нужно аккуратно сопоставлять имена слоёв.
    trainable_names = set(cfg.trainable_blocks)
    for name, module in G_train.named_modules():
        if any(t in name for t in trainable_names):
            for p in module.parameters():
                p.requires_grad = True

    clip_model, clip_preprocess = load_clip(cfg)
    return G_frozen, G_train, clip_model, clip_preprocess

### Training Script

This cell contains the main training loop as provided.

In [ ]:
import torch
from tqdm import tqdm
from src.config import Config
import src.model_setup # Import the module directly
import os
import importlib

# Reload src.model_setup to ensure the latest version with corrected imports is used
importlib.reload(src.model_setup)

def train(cfg: Config, sg_weights_path: str, output_dir: str):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    device = cfg.device
    G_frozen, G_train, clip_model, clip_preprocess = src.model_setup.setup_models(cfg, sg_weights_path)

    optimizer = torch.optim.Adam(
        [p for p in G_train.parameters() if p.requires_grad],
        lr=cfg.lr
    )

    source_emb = src.model_setup.get_clip_embeddings(clip_model, clip_preprocess, texts=[cfg.source_prompt])
    target_emb = src.model_setup.get_clip_embeddings(clip_model, clip_preprocess, texts=[cfg.target_prompt])
    direction = target_emb - source_emb

    for epoch in range(cfg.num_epochs):
        pbar = tqdm(range(100))  # для примера 100 шагов за эпоху
        for step in pbar:
            z = torch.randn(cfg.batch_size, cfg.latent_dim).to(device)

            # Генерируем картинку
            img = G_train(z, truncation=0.7)  # img: [B, 3, H, W], [0,1]

            # CLIP эмбеддинг картинки
            # clip_preprocess ожидает PIL или numpy; сделаем простой resize и нормализацию
            # Для продакшена лучше сделать отдельный transform, совместимый с CLIP
            clip_img = torch.nn.functional.interpolate(
                img, size=224, mode='bilinear', align_corners=False
            )
            clip_img = (clip_img - 0.5) / 0.5  # нормализация под CLIP
            # Теперь нужно привести к формату CLIP: [B, C, 224, 224], значения [-1,1] уже есть
            img_emb = src.model_setup.get_clip_embeddings(clip_model, None, images=clip_img)

            # Loss: косинусное расстояние до target, либо проекция на direction
            cos_sim = torch.cosine_similarity(img_emb, target_emb, dim=-1).mean()
            loss = -cos_sim  # максимизируем схожесть

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_description(f"Epoch {epoch}, loss={loss.item():.4f}")

        # Сохранение чекпоинта
        torch.save({
            "g_train_state_dict": G_train.state_dict(),
            "cfg": cfg,
        }, f"{output_dir}/epoch_{epoch}.pth")

# eval_generated function definition (copied from cell Sn6IFlrTe0Mt and adjusted for module imports)
def eval_generated(weights_frozen: str, weights_train: str, prompt: str, n: int = 8):
    cfg = Config() # Config is directly imported
    device = cfg.device

    G_frozen = src.model_setup.load_stylegan(cfg, weights_frozen)
    G_train  = src.model_setup.load_stylegan(cfg, weights_train)
    clip_model, clip_preprocess = src.model_setup.load_clip(cfg)

    target_emb = src.model_setup.get_clip_embeddings(clip_model, clip_preprocess, texts=[prompt])

    z = torch.randn(n, cfg.latent_dim).to(device)
    with torch.no_grad():
        img_frozen = G_frozen(z, truncation=0.7)
        img_train  = G_train(z,  truncation=0.7)

    # оценка схожести с таргет-промптом
    def cos_sim_to_target(imgs):
        clip_img = torch.nn.functional.interpolate(imgs, size=224, mode='bilinear', align_corners=False)
        clip_img = (clip_img - 0.5) / 0.5
        emb = src.model_setup.get_clip_embeddings(clip_model, None, images=clip_img)
        sim = torch.cosine_similarity(emb, target_emb, dim=-1)
        return sim.mean().item()

    sim_frozen = cos_sim_to_target(img_frozen)
    sim_train  = cos_sim_to_target(img_train)
    return sim_frozen, sim_train


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--weights", type=str, required=True)
    parser.add_argument("--output", type=str, default="./checkpoints")

    # In a Colab notebook, sys.argv is not populated, so parse_args() fails if required args are not provided.
    # We manually provide dummy arguments or you can replace these with actual paths.
    args = parser.parse_args(["--weights", "./checkpoints/dummy_weights.pth", "--output", "./checkpoints"])

    from src.config import Config
    cfg = Config()
    train(cfg, args.weights, args.output)

    print("\n--- Training complete. Starting evaluation ---")
    # Example usage after training creates checkpoints
    # Replace 'path/to/frozen_weights.pth' with your initial StyleGAN weights
    # Replace 'path/to/trained_weights.pth' with a checkpoint saved by the training script (e.g., ./checkpoints/epoch_X.pth)
    sim_frozen, sim_train = eval_generated(
        weights_frozen='./checkpoints/dummy_weights.pth', # This should be your pre-trained StyleGAN model path
        weights_train='./checkpoints/epoch_0.pth', # Replace with an actual trained checkpoint
        prompt='anime style face'
    )
    print(f"Cosine similarity for frozen model: {sim_frozen}")
    print(f"Cosine similarity for trained model: {sim_train}")

### Inference Script

This script defines an `inference` function to generate images from the trained StyleGAN model using a given prompt and weights.

In [ ]:
import torch
from src.config import Config
from src.model_setup import load_stylegan, load_clip, get_clip_embeddings

def inference(prompt: str, weights_path: str, num_samples: int = 4, seed: int = 0):
    cfg = Config()
    device = cfg.device
    torch.manual_seed(seed)

    G = load_stylegan(cfg, weights_path)  # это уже дообученный G_train
    clip_model, clip_preprocess = load_clip(cfg)

    target_emb = get_clip_embeddings(clip_model, clip_preprocess, texts=[prompt])

    z = torch.randn(num_samples, cfg.latent_dim).to(device)
    with torch.no_grad():
        imgs = G(z, truncation=0.7)

    # Если хочешь сделать «более направленную» генерацию, можно сделать оптимизацию z
    # (это ближе к StyleCLIP): оптимизируем z так, чтобы CLIP-эмбеддинг картинки был ближе к target_emb.
    return imgs

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", type=str, required=True)
    parser.add_argument("--weights", type=str, required=True)

    # Manually provide dummy arguments for Colab execution
    args = parser.parse_args(["--prompt", "a new image", "--weights", "./checkpoints/dummy_weights.pth"])

    imgs = inference(args.prompt, args.weights)
    # Дальше можно сохранить картинки через PIL

In [ ]:
!pip install clip
import torch
import clip
from torchvision import transforms
# from src.config import Config # Assuming Config is defined in an earlier cell

def load_clip(cfg: Config):
    model, preprocess = clip.load(cfg.clip_model, device=cfg.device, jit=False)
    model.eval()
    return model, preprocess

def get_clip_embeddings(model, preprocess, texts=None, images=None):
    if texts is not None:
        tokens = clip.tokenize(texts).to(model.device)
        with torch.no_grad():
            text_features = model.encode_text(tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)
        return text_features
    if images is not None:
        # images: B x C x H x W, значения [0, 1]
        norm_images = preprocess(images).to(model.device)  # тут может потребоваться цикл
        with torch.no_grad():
            image_features = model.encode_image(norm_images)
            image_features /= image_features.norm(dim=-1, keepdim=True)
        return image_features

# Здесь ты подгружаешь StyleGAN-2.
# Для простоты возьмём готовый репозиторий stylegan2-pytorch (github.com/rosinality/stylegan2-pytorch)
# и импортируем Generator.
def load_stylegan(cfg: Config, weights_path: str):
    # Need to install stylegan2-pytorch if not already present
    try:
        from stylegan2_pytorch import Generator
    except ImportError:
        print("Installing stylegan2-pytorch...")
        !pip install stylegan2-pytorch
        from stylegan2_pytorch import Generator

    G = Generator(cfg.image_size, cfg.latent_dim, 8).to(cfg.device)
    ckpt = torch.load(weights_path, map_location=cfg.device)
    G.load_state_dict(ckpt["g_ema"])
    G.eval()
    return G

def setup_models(cfg: Config, sg_weights: str):
    device = cfg.device
    G_frozen = load_stylegan(cfg, sg_weights)
    G_train = load_stylegan(cfg, sg_weights)

    # Заморозим все параметры G_frozen
    for p in G_frozen.parameters():
        p.requires_grad = False

    # Разморозим только нужные блоки в G_train
    def freeze_all(m):
        for p in m.parameters():
            p.requires_grad = False
    freeze_all(G_train)

    # Тут нужно пройтись по модулям и включить grad для нужных имён.
    # Это упрощённый пример; в реальности нужно аккуратно сопоставлять имена слоёв.
    trainable_names = set(cfg.trainable_blocks)
    for name, module in G_train.named_modules():
        if any(t in name for t in trainable_names):
            for p in module.parameters():
                p.requires_grad = True

    clip_model, clip_preprocess = load_clip(cfg)
    return G_frozen, G_train, clip_model, clip_preprocess

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

In [ ]:
import torch
from src.config import Config
from src.model_setup import load_stylegan, load_clip, get_clip_embeddings

def eval_generated(weights_frozen: str, weights_train: str, prompt: str, n: int = 8):
    cfg = Config()
    device = cfg.device

    G_frozen = load_stylegan(cfg, weights_frozen)
    G_train  = load_stylegan(cfg, weights_train)
    clip_model, clip_preprocess = load_clip(cfg)

    target_emb = get_clip_embeddings(clip_model, clip_preprocess, texts=[prompt])

    z = torch.randn(n, cfg.latent_dim).to(device)
    with torch.no_grad():
        img_frozen = G_frozen(z, truncation=0.7)
        img_train  = G_train(z,  truncation=0.7)

    # оценка схожести с таргет-промптом
    def cos_sim_to_target(imgs):
        clip_img = torch.nn.functional.interpolate(imgs, size=224, mode='bilinear', align_corners=False)
        clip_img = (clip_img - 0.5) / 0.5
        emb = get_clip_embeddings(clip_model, None, images=clip_img)
        sim = torch.cosine_similarity(emb, target_emb, dim=-1)
        return sim.mean().item()

    sim_frozen = cos_sim_to_target(img_frozen)
    sim_train  = cos_sim_to_target(img_train)
    return sim_frozen, sim_train

In [ ]:
!pip install datasets pillow torch torchvision tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/clip_stylegan_project"
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

In [ ]:
from datasets import load_dataset
from PIL import Image
import torch
from torchvision import transforms
from tqdm import tqdm
import os

target_dir = "data/real_faces_utk"
os.makedirs(target_dir, exist_ok=True)

print("Загружаем UTKFace...")
dataset = load_dataset("nielsr/utkface", split="train") # Reverting to canonical dataset name

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # [0,255] -> [0,1], C,H,W
])

print("Обрабатываем и сохраняем (первые 2000 изображений для старта)…")
save_count = 0
max_images = 2000

for item in tqdm(dataset, total=max_images):
    if save_count >= max_images:
        break
    try:
        img = item["image"]
        if isinstance(img, str):
            img = Image.open(img).convert("RGB")
        elif isinstance(img, Image.Image):
            img = img.convert("RGB")
        else:
            continue

        img_t = transform(img)
        save_path = os.path.join(target_dir, f"real_{save_count:05d}.png")
        pil_img = transforms.ToPILImage()(img_t.clamp(0, 1))
        pil_img.save(save_path)
        save_count += 1
    except Exception:
        continue

print(f"Готово: сохранено {save_count} изображений в {target_dir}")

### Verify StyleGAN Checkpoint File

Run the following command to list the contents of the `./checkpoints/` directory and ensure `ffhq_stylegan2_pytorch.pt` is present. Remember that your working directory is set to `/content/drive/MyDrive/clip_stylegan_project`.

In [ ]:
!ls -lh ./checkpoints/

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

In [ ]:
#@title Setup
!pip install pydrive

import os

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

pretrained_model_dir = os.path.join("/content", "models")
os.makedirs(pretrained_model_dir, exist_ok=True)

restyle_dir = os.path.join("/content", "restyle")
stylegan_ada_dir = os.path.join("/content", "stylegan_ada")
stylegan_nada_dir = os.path.join("/content", "stylegan_nada")

output_dir = os.path.join("/content", "output")

output_model_dir = os.path.join(output_dir, "models")
output_image_dir = os.path.join(output_dir, "images")

download_with_pydrive = True #@param {type:"boolean"}

class Downloader(object):
    def __init__(self, use_pydrive):
        self.use_pydrive = use_pydrive

        if self.use_pydrive:
            self.authenticate()

    def authenticate(self):
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        self.drive = GoogleDrive(gauth)

    def download_file(self, file_id, file_dst):
        if self.use_pydrive:
            downloaded = self.drive.CreateFile({'id':file_id})
            downloaded.FetchMetadata(fetch_all=True)
            downloaded.GetContentFile(file_dst)
        else:
            !gdown --id $file_id -O $file_dst

downloader = Downloader(download_with_pydrive)

# install requirements
!git clone https://github.com/yuval-alaluf/restyle-encoder.git $restyle_dir

!wget https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip
!sudo unzip ninja-linux.zip -d /usr/local/bin/
!sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force

!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

!git clone https://github.com/NVlabs/stylegan2-ada/ $stylegan_ada_dir
!git clone https://github.com/rinongal/stylegan-nada.git $stylegan_nada_dir

from argparse import Namespace

import sys
import numpy as np

from PIL import Image

import torch
import torchvision.transforms as transforms

sys.path.append(restyle_dir)
sys.path.append(stylegan_nada_dir)
sys.path.append(os.path.join(stylegan_nada_dir, "ZSSGAN"))

device = 'cuda'

In [ ]:
!pip install datasets huggingface_hub pillow torch torchvision tqdm omegaconf -q

from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/clip_stylegan_project"
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

print("Среда готова, работаем в:", os.getcwd())

In [ ]:
from datasets import load_dataset
from PIL import Image
import torch
from torchvision import transforms
from tqdm import tqdm
import os

data_dir = "data/real_faces_utk"
os.makedirs(data_dir, exist_ok=True)

# Check for Hugging Face token for authentication
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    print("Warning: Hugging Face token (HF_TOKEN) not found. Some datasets may require authentication.")
    print("Please visit https://huggingface.co/settings/tokens to create a token and add it to Colab secrets as 'HF_TOKEN'.")

print("Загружаем UTKFace (первые 1000 изображений)...")
# Pass the token to load_dataset
dataset = load_dataset("HuggingFaceM4/UTKFace", split="train", token=hf_token)

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # [0,255] -> [0,1], C,H,W
])

save_count = 0
max_images = 1000

for item in tqdm(dataset, total=max_images):
    if save_count >= max_images:
        break
    try:
        img = item["image"]
        if isinstance(img, str):
            img = Image.open(img).convert("RGB")
        elif isinstance(img, Image.Image):
            img = img.convert("RGB")
        else:
            continue

        img_t = transform(img)
        save_path = os.path.join(data_dir, f"real_{save_count:05d}.png")
        pil_img = transforms.ToPILImage()(img_t.clamp(0, 1))
        pil_img.save(save_path)
        save_count += 1
    except Exception:
        continue

print(f"Готово: {save_count} изображений в {data_dir}")

In [ ]:
import os

data_dir = "data/real_faces_utk"

if os.path.exists(data_dir):
    files_in_dir = os.listdir(data_dir)
    png_files = [f for f in files_in_dir if f.endswith('.png')]
    print(f"Найдено {len(png_files)} PNG файлов в директории '{data_dir}':")
    # Display first few files to confirm
    for i, file_name in enumerate(png_files[:5]):
        print(f"- {file_name}")
    if len(png_files) > 5:
        print("...и еще {len(png_files) - 5} файлов")
else:
    print(f"Директория '{data_dir}' не найдена.")

In [ ]:
import os

data_dir = "data/real_faces_utk"

if os.path.exists(data_dir):
    print(f"Contents of '{data_dir}':")
    for item in os.listdir(data_dir):
        print(f"- {item}")
else:
    print(f"Директория '{data_dir}' не найдена.")

In [ ]:
import os

hf_token_value = os.environ.get("HF_TOKEN")

if hf_token_value:
    print("HF_TOKEN успешно подключен!")
    # For security, avoid printing the full token in output,
    # but you can check its presence.
    print(f"Длина токена: {len(hf_token_value)} символов")
else:
    print("HF_TOKEN не найден. Пожалуйста, убедитесь, что вы добавили его в Colab Secrets.")

In [ ]:
import os, torch
from PIL import Image
from torchvision import transforms
import clip
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Используем устройство:", device)

model, _ = clip.load("ViT-B/32", device=device, jit=False)
model.eval()

data_dir = "data/real_faces_utk"
files = [f for f in os.listdir(data_dir) if f.endswith(".png")]
files = files[:500]  # берём 500 для скорости

print(f"Считаем усреднённый CLIP‑эмбеддинг по {len(files)} реальным лицам...")

batch_size = 32
all_feats = []

transform_clip = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
])

for i in tqdm(range(0, len(files), batch_size)):
    batch_paths = files[i:i+batch_size]
    batch_imgs = []
    for fname in batch_paths:
        path = os.path.join(data_dir, fname)
        img = Image.open(path).convert("RGB")
        img = transform_clip(img)
        batch_imgs.append(img)

    batch_tensor = torch.stack([transforms.ToTensor()(img) for img in batch_imgs]).to(device)
    batch_tensor = (batch_tensor - 0.5) / 0.5  # CLIP ждёт [-1,1]

    with torch.no_grad():
        feats = model.encode_image(batch_tensor)
        feats /= feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats)

target_emb = torch.cat(all_feats, dim=0).mean(dim=0, keepdim=True)  # усредняем
torch.save(target_emb, "models/real_faces_clip_emb.pt")
print("Усреднённый embedding сохранён: models/real_faces_clip_emb.pt")

In [ ]:
import gdown, os

os.makedirs("models", exist_ok=True)
weights_path = "models/stylegan2-ffhq-256.pth"

if not os.path.exists(weights_path):
    print("Скачиваем веса StyleGAN2 FFHQ 256...")
    # Официальная ссылка от rosinality (проверенная)
    gdown.download("https://drive.google.com/uc?id=1Sh4pkQ0LWOCSBYcS9uIW9ap468ZQFn-o", weights_path, quiet=False)
else:
    print("Веса уже есть.")

In [ ]:
import torch, os
from tqdm import tqdm
from stylegan2_pytorch.stylegan2_pytorch import Generator # Corrected import
import clip # Added import
from torchvision import transforms # Added import

device = "cuda" if torch.cuda.is_available() else "cpu"
latent_dim, image_size, batch_size = 512, 256, 4
lr, num_epochs, steps_per_epoch = 2e-4, 3, 150  # маленький эксперимент: 3 эпохи по 150 шагов

# Инициализация моделей
G_frozen = Generator(image_size, latent_dim, 8).to(device)
G_train = Generator(image_size, latent_dim, 8).to(device)

ckpt = torch.load("models/stylegan2-ffhq-256.pth", map_location=device)
G_frozen.load_state_dict(ckpt["g_ema"])
G_train.load_state_dict(ckpt["g_ema"])

# Замораживаем всё
for p in G_frozen.parameters(): p.requires_grad = False
for p in G_train.parameters(): p.requires_grad = False

# Размораживаем только to_rgb
for name, module in G_train.named_modules():
    if "to_rgb" in name:
        for p in module.parameters():
            p.requires_grad = True

optimizer = torch.optim.Adam([p for p in G_train.parameters() if p.requires_grad], lr=lr)

# Загружаем таргет-эмбеддинг (реальные лица)
target_emb = torch.load("models/real_faces_clip_emb.pt", map_location=device).to(device)
# Source-эмбеддинг сделаем нейтральным (нулевой вектор) — тогда проекция будет просто «насколько близко к target»
source_emb = torch.zeros_like(target_emb)
direction = target_emb - source_emb
direction /= direction.norm(dim=-1, keepdim=True) # Corrected typo: keepim -> keepdim

# CLIP для оценки
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device, jit=False)
clip_model.eval()

def get_clip_img_emb(imgs):
    # imgs: [B,3,256,256], [0,1] -> ресайз 224, норма [-1,1]
    B = imgs.shape[0]
    clip_imgs = []
    for i in range(B):
        img = transforms.functional.resize(imgs[i], 224)
        img = transforms.functional.center_crop(img, 224)
        clip_imgs.append((img - 0.5) / 0.5)
    clip_imgs = torch.stack(clip_imgs).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(clip_imgs)
        emb /= emb.norm(dim=-1, keepdim=True)
    return emb

logs = []
os.makedirs("checkpoints/exp_real_faces", exist_ok=True)

for epoch in range(num_epochs):
    pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch}")
    for step in pbar:
        z = torch.randn(batch_size, latent_dim).to(device)

        with torch.no_grad():
            img_frozen = G_frozen(z, truncation=0.7)
        img_train = G_train(z, truncation=0.7)

        emb_train = get_clip_img_emb(img_train)
        # Проекция на направление target
        proj = torch.cosine_similarity(emb_train - source_emb, direction, dim=-1).mean()
        loss_clip = -proj

        # Регуляризация (чтобы не улетал от frozen)
        reg_loss = ((img_train - img_frozen) ** 2).mean()
        loss = loss_clip + 0.1 * reg_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix({"loss": f"{loss.item():.4f}", "proj": f"{proj.item():.4f}"})
        logs.append({"epoch": epoch, "step": step, "loss": loss.item(), "proj": proj.item()})

    torch.save({
        "g_train_state_dict": G_train.state_dict(),
        "logs": logs,
    }, f"checkpoints/exp_real_faces/epoch_{epoch}.pth")

print("Обучение завершено. Чекпоинт сохранён.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

torch.manual_seed(42)
z = torch.randn(4, latent_dim).to(device)

# Provide a dummy input_noise tensor with a 4D shape (batch, height, width, channels)
# The Generator.forward() method in this environment seems to require it as a positional argument
# and expects it to be a spatial noise tensor.
dummy_input_noise = torch.randn(z.shape[0], image_size, image_size, 1).to(device)

with torch.no_grad():
    img_frozen = G_frozen(z, dummy_input_noise).cpu()
    img_train = G_train(z, dummy_input_noise).cpu()

emb_frozen = get_clip_img_emb(img_frozen.to(device))
emb_train = get_clip_img_emb(img_train.to(device))

sim_frozen = torch.cosine_similarity(emb_frozen, target_emb, dim=-1).mean().item()
sim_train = torch.cosine_similarity(emb_train, target_emb, dim=-1).mean().item()

print(f"Среднее CLIP-сходство с таргет-эмбеддингом (реальные лица):")
print(f"  G_frozen (до):  {sim_frozen:.4f}")
print(f"  G_train (после): {sim_train:.4f}")
print(f"  Рост: {sim_train - sim_frozen:.4f}")

def tensor_to_pil(t):
    t = t.clamp(0, 1).permute(0, 2, 3, 1).mul(255).byte().numpy()
    return [Image.fromarray(t[i]) for i in range(t.shape[0])]

pil_frozen = tensor_to_pil(img_frozen)
pil_train = tensor_to_pil(img_train)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(pil_frozen[i])
    axes[0, i].axis("off")
    axes[0, i].set_title("G_frozen")
    axes[1, i].imshow(pil_train[i])
    axes[1, i].axis("off")
    axes[1, i].set_title("G_train")
plt.tight_layout()
plt.show()

# Сохраняем коллаж для отчёта
collage = Image.new("RGB", (4*256, 2*256))
for i in range(4):
    collage.paste(pil_frozen[i], (i*256, 0))
    collage.paste(pil_train[i], (i*256, 256))
collage.save("results/exp_real_faces_frozen_vs_train.png")
print("Коллаж сохранён: results/exp_real_faces_frozen_vs_train.png")